# Retrieval-Augmented Generation (RAG)

This notebook demonstrates a simplified RAG pipeline:
1. Create a document knowledge base
2. Split documents into chunks
3. Generate embeddings
4. Perform similarity search
5. Retrieve relevant context
6. Build a prompt for an LLM

In [1]:
# Libraries
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# Create knowledge base
documents = [
"""
Free shipping increases conversion rates for orders above $50.
Customers are more likely to complete purchases when shipping costs are removed.
""",

"""
Product return policies improve customer trust.
A 30-day return window reduces purchase hesitation.
""",

"""
Seasonal promotions significantly increase sales.
Events such as Black Friday or Cyber Monday generate spikes in demand.
""",

"""
High product ratings and reviews increase marketplace credibility
and help users make purchasing decisions.
"""
]

In [3]:
# Chunking 
def chunk_text(text, chunk_size=80):
    words = text.split()
    chunks = []
    
    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)
        
    return chunks


chunks = []

for doc in documents:
    chunks.extend(chunk_text(doc))

chunks

['Free shipping increases conversion rates for orders above $50. Customers are more likely to complete purchases when shipping costs are removed.',
 'Product return policies improve customer trust. A 30-day return window reduces purchase hesitation.',
 'Seasonal promotions significantly increase sales. Events such as Black Friday or Cyber Monday generate spikes in demand.',
 'High product ratings and reviews increase marketplace credibility and help users make purchasing decisions.']

In [4]:
# Create embedgings
model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = model.encode(chunks)

len(chunk_embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

4

In [5]:
# User Query

query = "How do promotions affect sales?"

query_embedding = model.encode([query])

In [6]:
# Similarioty Search 
similarities = cosine_similarity(query_embedding, chunk_embeddings)[0]

similarities

array([0.38601863, 0.29429108, 0.63687956, 0.45854473], dtype=float32)

In [7]:
# Retrieve top-k chunks
top_k = 3

top_indices = similarities.argsort()[-top_k:][::-1]

retrieved_chunks = [chunks[i] for i in top_indices]

retrieved_chunks

['Seasonal promotions significantly increase sales. Events such as Black Friday or Cyber Monday generate spikes in demand.',
 'High product ratings and reviews increase marketplace credibility and help users make purchasing decisions.',
 'Free shipping increases conversion rates for orders above $50. Customers are more likely to complete purchases when shipping costs are removed.']

In [8]:
# Ranking Visualization
results = pd.DataFrame({
    "chunk": chunks,
    "similarity": similarities
})

results.sort_values("similarity", ascending=False).head(5)

,chunk,similarity
2,Seasonal promotions significantly increase sal...,0.636880
3,High product ratings and reviews increase mark...,0.458545
0,Free shipping increases conversion rates for o...,0.386019
1,Product return policies improve customer trust...,0.294291


In [9]:
# Build augmented prompt

context = "\n".join(retrieved_chunks)

prompt = f"""
Context:
{context}

Question:
{query}

Answer using the information provided in the context.
"""

print(prompt)


Context:
Seasonal promotions significantly increase sales. Events such as Black Friday or Cyber Monday generate spikes in demand.
High product ratings and reviews increase marketplace credibility and help users make purchasing decisions.
Free shipping increases conversion rates for orders above $50. Customers are more likely to complete purchases when shipping costs are removed.

Question:
How do promotions affect sales?

Answer using the information provided in the context.



In [10]:
print("""
RAG Pipeline Explained:

1. Documents are split into chunks.
2. Each chunk is converted into an embedding.
3. The user query is also embedded.
4. Similarity search retrieves the most relevant chunks.
5. Retrieved context is added to the prompt.
6. An LLM generates an answer using that context.
""")


RAG Pipeline Explained:

1. Documents are split into chunks.
2. Each chunk is converted into an embedding.
3. The user query is also embedded.
4. Similarity search retrieves the most relevant chunks.
5. Retrieved context is added to the prompt.
6. An LLM generates an answer using that context.



In [11]:
# Potential production improvements:
#
# - Use a vector database (FAISS, Pinecone, Weaviate)
# - Store millions of document chunks
# - Connect to a real LLM (OpenAI, Llama, Mistral)
# - Implement evaluation and monitoring